[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 01](README.md)

# Modelos de paralelismo y descomposición

**Tema:** 01 · **Sesiones:** 3 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Qué parte del trabajo puede ejecutarse simultáneamente y qué dependencias fijan el camino crítico?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** La primera decisión paralela no es cuántos hilos usar, sino cómo expresar trabajo y dependencias. El DAG permite razonar antes de elegir una API.

**Prerrequisitos.**

- Aritmética, funciones y lectura de gráficas.
- Python básico para modificar parámetros y ejecutar aserciones.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Distinguir concurrencia, paralelismo de datos y de tareas.
- Representar dependencias mediante un DAG.
- Calcular trabajo, span y paralelismo promedio.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

El trabajo T1 suma el costo de todas las tareas; el span T∞ es el camino dependiente más largo.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

La aceleración con p recursos está acotada por min(p, T1/T∞), aun sin costos de comunicación.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Una descomposición correcta conserva dependencias y evita crear más coordinación que cómputo útil.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- trabajo T₁ — costo total de todas las operaciones
- span T∞ — costo del camino dependiente más largo
- eficiencia — aceleración dividida entre recursos
- intensidad aritmética — FLOP realizados por byte movido


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Dag Camino Critico

![DAG con trabajo y camino crítico](../../images/dag-camino-critico.svg)

**Cómo leerlo.** El trazo destacado une las tareas que determinan el span. La rama B puede terminar antes sin reducir el total mientras A siga siendo más larga.

### Distribucion Trabajo

![Iteraciones distribuidas y reducción final](../../images/distribucion-trabajo.svg)

**Cómo leerlo.** Verifica dos propiedades: cada iteración pertenece a un trabajador y la combinación de parciales reproduce la referencia serial.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "01"
NOTEBOOK = "01_fundamentos/01_modelos.ipynb"
assert (ROOT / "curso" / "notebooks" / "01_fundamentos" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Trabajo y camino crítico

**Situación.** Se evalúa un DAG pequeño con duraciones explícitas.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
tasks = {"leer": 2, "partir": 1, "A": 5, "B": 4, "combinar": 2}
predecessors = {"leer": [], "partir": ["leer"], "A": ["partir"], "B": ["partir"], "combinar": ["A", "B"]}
finish = {}
for task in tasks:
    start = max((finish[p] for p in predecessors[task]), default=0)
    finish[task] = start + tasks[task]
work = sum(tasks.values())
span = max(finish.values())
parallelism = work / span
assert (work, span) == (14, 10)
print({"work": work, "span": span, "parallelism": round(parallelism, 2), "finish": finish})


### Explicación del resultado

A y B son concurrentes, pero lectura, partición y combinación permanecen en el camino crítico.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Partición balanceada

**Situación.** Se distribuyen n elementos sin perder ni duplicar índices.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
def ranges(n, workers):
    q, r = divmod(n, workers)
    start = 0
    result = []
    for worker in range(workers):
        size = q + (worker < r)
        result.append((start, start + size))
        start += size
    return result
chunks = ranges(23, 4)
covered = [i for begin, end in chunks for i in range(begin, end)]
assert covered == list(range(23))
print(chunks)


### Lectura razonada

El resto se reparte de forma determinista y la cobertura constituye una prueba simple de corrección.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. Si la tarea B durara 7 unidades, ¿qué cambiaría en el span y en el camino crítico?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Dibujar el DAG de una operación del curso.
2. Identificar T1, T∞ y la granularidad de cada nodo.
3. Proponer una descomposición y señalar la sincronización necesaria.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.

### Actividades compilables relacionadas

- [Ejercicio C17: partición balanceada y reducción](../../ejercicios/01_fundamentos/01_particion_reduccion/README.md)


## Errores frecuentes

- Confundir más tareas con mayor paralelismo.
- Omitir dependencias de datos.
- Evaluar únicamente el tiempo paralelo sin referencia serial.


## Criterios de aceptación

- DAG acíclico y dependencias justificadas.
- Cobertura de datos sin solapamientos involuntarios.
- Cotas de aceleración calculadas antes de medir.


## Síntesis

- La pregunta que debes poder responder es: **¿Qué parte del trabajo puede ejecutarse simultáneamente y qué dependencias fijan el camino crítico?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Planeación: fundamentos](../../../docs/PLANEACION_CURSO.md#6-calendario-de-38-sesiones)
- [Índice del curso](../../../INDICE_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 01](README.md)
